# Chart Data Extraction Pipeline

This notebook extracts numerical values from bar chart images.

Pipeline:
1. Object detection (YOLO) → detect bars, legends, plot area
2. OCR JSON parsing → extract text blocks
3. Axis & tick detection
4. Pixel-to-value ratio estimation
5. Legend color matching (deep features)
6. Bar-to-legend assignment
7. Value computation
8. Export results to Excel / JSON

### 1. Install Libraries


In [33]:
!pip install xlsxwriter

import xlsxwriter
import cv2, imutils, re, sys, math
import json, os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from matplotlib import rcParams

import pandas as pd

In [34]:
!pip install ultralytics -q

from ultralytics import YOLO
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # tránh một số lỗi lib

In [35]:
from google.colab import drive
drive.mount('/content/drive/')


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


### 2. Configuration

Define:
- Dataset directories
- Model weight paths
- Output directory

Switch between:
- Google Colab
- Local machine

In [36]:
# ==== CONFIG  ====
USE_COLAB = True

if USE_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/data_ex_v7/dataset")
    WEIGHT_PATH = "/content/drive/MyDrive/data_ex_v7/yolov8s_exp4/weights/best.pt"
    OUT_DIR = Path("/content/drive/MyDrive/data_ex_v7/outputs")
else:
    BASE_DIR = Path("D:/Projects/.../dataset")
    WEIGHT_PATH = "D:/Projects/.../weights/best.pt"
    OUT_DIR = Path("D:/Projects/.../outputs")
# ==================================
IMG_DIR  = BASE_DIR / "images_debug"
JSON_DIR = BASE_DIR / "json"
EXCEL_PATH = OUT_DIR / "y_values.xlsx"
JSON_PATH  = OUT_DIR / "y_values.json"
# ======================


## 1. Text, Label Extraction And Axis Analysis Task:


This section extracts text labels from OCR JSON (input from previous task) and classifies them into:

- X tick labels
- Y tick labels
- Axis titles
- Legend labels

It uses:
- Bounding box geometry
- Axis orientation
- Cross-product position test

In [37]:
def lineIntersectsRectX(candx, rect):
    # Check whether a vertical line at x = candx intersects the rectangle
    (x, y, w, h) = rect

    if x <= candx <= x + w:
        return True
    else:
        return False


def lineIntersectsRectY(candy, rect):
    # Check whether a horizontal line at y = candy intersects the rectangle
    (x, y, w, h) = rect

    if y <= candy <= y + h:
        return True
    else:
        return False


def cleanText(image_text):
    # Remove noisy OCR text (e.g., single character 'I')
    return [
        (text, (textx, texty, w, h))
        for text, (textx, texty, w, h) in image_text
        if text.strip() != 'I'
    ]


def point_line_distance(px, py, x1, y1, x2, y2):
    """
    Compute distance from point (px, py) to a line passing through (x1, y1) - (x2, y2)
    """
    return abs((y2 - y1) * px - (x2 - x1) * py + x2 * y1 - y2 * x1) / \
           np.hypot(y2 - y1, x2 - x1)


def getProbableLabels(image, d, xaxis, yaxis):
    """
    Parameters
    ----------
    image : RGB image (H, W, 3)
    d     : JSON object loaded using json.loads(...)
    xaxis, yaxis : tuple (x1, y1, x2, y2)

    Requirements
    ------------
    - d["task3"]["input"]["task2_output"]["text_blocks"]
    - d["task3"]["output"]["text_roles"]
      roles include: tick_label, axis_title, legend_label
    """

    # ===== 1) Extract text_blocks from task3.input.task2_output =====
    try:
        text_blocks = d["task3"]["input"]["task2_output"]["text_blocks"]
    except KeyError:
        # fallback to task2 output if task3 structure is unavailable
        text_blocks = d["task2"]["output"]["text_blocks"]

    id_to_text = {}
    id_to_rect = {}
    raw_image_text = []

    for block in text_blocks:
        bid = block["id"]
        txt = block["text"]
        poly = block["polygon"]

        xs = [poly["x0"], poly["x1"], poly["x2"], poly["x3"]]
        ys = [poly["y0"], poly["y1"], poly["y2"], poly["y3"]]
        x_min, y_min = min(xs), min(ys)
        w = max(xs) - x_min
        h = max(ys) - y_min

        id_to_text[bid] = txt
        id_to_rect[bid] = (x_min, y_min, w, h)
        raw_image_text.append((txt, (x_min, y_min, w, h)))

    # Clean OCR text
    image_text = cleanText(raw_image_text)

    # ===== 2) Extract text_roles from task3.output =====
    text_roles = d["task3"]["output"]["text_roles"]
    id_to_role = {item["id"]: item["role"] for item in text_roles}

    # ===== 3) Split text by role: tick_label, axis_title, legend_label =====
    tick_blocks   = []  # [(text, rect), ...]
    axis_blocks   = []  # [(text, rect), ...]
    legend_blocks = []  # [(text, rect), ...]

    for bid, role in id_to_role.items():
        if bid not in id_to_text:
            continue

        text = id_to_text[bid]
        rect = id_to_rect[bid]

        if role == "tick_label":
            tick_blocks.append((text, rect))
        elif role == "axis_title":
            axis_blocks.append((text, rect))
        elif role == "legend_label":
            legend_blocks.append((text, rect))
        else:
            # ignore other roles
            pass

    # ===== 4) Split tick labels into Y-ticks and X-ticks using cross product =====
    (x1,  y1,  x2,  y2)  = xaxis
    (yx1, yy1, yx2, yy2) = yaxis

    x_tick_list = []
    y_tick_list = []

    for text, (tx, ty, w, h) in tick_blocks:
        # use bbox center for stability
        cx = tx + w / 2.0
        cy = ty + h / 2.0

        side_xaxis = np.sign((x2  - x1)  * (cy - y1)  - (y2  - y1)  * (cx - x1))
        side_yaxis = np.sign((yx2 - yx1) * (cy - yy1) - (yy2 - yy1) * (cx - yx1))

        # left of Y-axis and above X-axis → Y tick
        if side_yaxis == 1:
            y_tick_list.append((text, (tx, ty, w, h)))

        # right of Y-axis and below X-axis → X tick
        elif side_xaxis == 1 and side_yaxis == -1:
            x_tick_list.append((text, (tx, ty, w, h)))

    # ===== 5) Split axis_title into X-axis title and Y-axis title =====
    x_title = []
    y_title = []

    for text, (tx, ty, w, h) in axis_blocks:
        cx = tx + w / 2.0
        cy = ty + h / 2.0

        dist_to_x = point_line_distance(cx, cy, x1, y1, x2, y2)
        dist_to_y = point_line_distance(cx, cy, yx1, yy1, yx2, yy2)

        # closer to Y-axis → Y title
        if dist_to_y < dist_to_x:
            y_title.append((text, (tx, ty, w, h)))
        else:
            # closer to X-axis → X title
            x_title.append((text, (tx, ty, w, h)))

    # ===== 6) Legend labels =====
    # legends = [text for (text, rect) in legend_blocks]
    # keep format [(text, rect), ...]
    legend_text_boxes = legend_blocks[:]

    # ===== 7) Return results (compatible with previous format) =====
    return (
        image,
        x_tick_list,
        x_title,
        y_tick_list,
        y_title,
        legend_text_boxes,
        image_text,
    )

### 2. Pixel-to-Value Ratio Estimation

Convert pixel height → actual numeric value.

Steps:
1. Extract numeric values from Y-axis ticks
2. Sort ticks by position
3. Compute pixel differences
4. Compute numeric differences
5. Estimate ratio

value = min_val + pixel_height * ratio

In [38]:
# ============ Normal version, don't work for float number, scientific symbol

# def reject_outliers(data, m=1):
#     return data[abs(data - np.mean(data)) <= m * np.std(data)]

# def getRatio(path, y_tick_list, xaxis, yaxis):
#     # y_labels_list = []  # [(text, rect), ...]
#     list_text = []
#     list_ticks = []

#     # filepath = img_dir + "/" + path.name
#     filepath = path
#     image = cv2.imread(filepath)
#     image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     height, width, channels = image.shape

#     (x1, y1, x2, y2) = xaxis
#     (x11, y11, x22, y22) = yaxis

#     # for text, (textx, texty, w, h) in y_tick_list:
#     #     text = text.strip()

#     #     # Consider numeric only for ticks on y-axis
#     #     numbers = re.findall(r'\d+(?:\.\d+)?', text)
#     #     if bool(numbers):
#     #         list_text.append((numbers[0], (textx, texty, w, h)))


#     for text, (textx, texty, w, h) in y_tick_list: # Hoặc y_labels_list
#         text = text.strip()
#         # Regex mới bao quát mọi trường hợp
#         pattern = r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?'
#         numbers = re.findall(pattern, text)

#         if bool(numbers):
#             try:
#                 best_match = max(numbers, key=len)
#                 val = float(best_match)
#                 list_text.append(val)
#                 list_ticks.append(float(texty + h))
#             except ValueError:
#                 continue # Bỏ qua nếu không convert được

#     # Get the y-labels by finding the maximum
#     # intersections with the sweeping line
#     maxIntersection = 0
#     maxList = []
#     for i in range(x11):
#         count = 0
#         current = []
#         for index, (text, rect) in enumerate(list_text):
#             if lineIntersectsRectX(i, rect):
#                 count += 1
#                 current.append(list_text[index])

#         if count > maxIntersection:
#             maxIntersection = count
#             maxList = current

#     # Get list of text and ticks
#     list_text = []
#     for text, (textx, texty, w, h) in maxList:
#         list_text.append(float(text))
#         list_ticks.append(float(texty + h))

#     text_sorted = (sorted(list_text))
#     ticks_sorted  = (sorted(list_ticks))

#     ticks_diff = ([ticks_sorted[i] - ticks_sorted[i-1] for i in range(1, len(ticks_sorted))])
#     text_diff = ([text_sorted[i] - text_sorted[i-1] for i in range(1, len(text_sorted))])
#     # print("[get text-to-tick ratio] ticks_diff: {0}, text_diff: {1}".format(ticks_diff, text_diff))

#     # Detected text may not be perfect! Remove the outliers.
#     ticks_diff = reject_outliers(np.array(ticks_diff), m=1)
#     text_diff = reject_outliers(np.array(text_diff), m=1)
#     # print("[reject_outliers] ticks_diff: {0}, text_diff: {1}".format(ticks_diff, text_diff))

#     normalize_ratio = np.array(text_diff).mean() / np.array(ticks_diff).mean()

#     # Lấy giá trị nhỏ nhất tìm thấy làm điểm neo (anchor)
#     min_val = text_sorted[0]
#     min_pixel = ticks_sorted[0] # Lưu ý: ticks_sorted cần khớp chiều với text

#     return text_sorted, normalize_ratio, (min_val, min_pixel)

#     # return text_sorted, normalize_ratio

In [39]:
def infer_ndigits_from_ticks(y_tick_list, default=1, cap=3):
    pattern = r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?'
    decs = []
    for text, _ in y_tick_list:
        text = text.strip()
        nums = re.findall(pattern, text)
        if not nums:
            continue
        s = max(nums, key=len)

        # scientific notation
        if 'e' in s.lower():
            return min(cap, max(default, 2))

        if '.' in s:
            frac = s.split('.', 1)[1]
            frac = frac.split('e', 1)[0].split('E', 1)[0]
            decs.append(len(frac.rstrip('0')))
        else:
            decs.append(0)

    if not decs:
        return default
    return min(cap, max(decs))

In [40]:
def reject_outliers(data, m=1):
    # Remove values that are farther than m standard deviations from the mean
    return data[abs(data - np.mean(data)) <= m * np.std(data)]


def getRatio_optimized(y_tick_list):
    # y_tick_list = [(text, rect), ...] already filtered as Y-axis ticks

    list_text = []
    list_ticks = []

    # 1. Extract numeric values and corresponding Y coordinates
    for text, (textx, texty, w, h) in y_tick_list:
        text = text.strip()

        # --- OPTIONAL PREPROCESSING ---
        # Remove noisy characters if needed (e.g., thousand separators)
        # text = text.replace(',', '')

        # Regex that supports integers, floats, and scientific notation
        pattern = r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?'
        numbers = re.findall(pattern, text)

        if bool(numbers):
            # Python float() automatically handles scientific notation
            # e.g., '1.0E-03' -> 0.001
            try:
                # Choose the longest match (more reliable for OCR noise)
                # Example: "Ver 1.0E-02" -> regex finds multiple numbers
                # prefer the most informative one
                best_match = max(numbers, key=len)
                val = float(best_match)

                list_text.append(val)

                # Use bottom of bounding box as tick position
                list_ticks.append(float(texty + h))

            except ValueError:
                # Skip if conversion fails
                continue

    # If fewer than 2 ticks, ratio cannot be computed
    if len(list_text) < 2:
        return sorted(list_text), 0

    # 2. Sort values to ensure consecutive differences are meaningful
    text_sorted = sorted(list_text)
    ticks_sorted = sorted(list_ticks)

    # 3. Compute differences between consecutive ticks
    ticks_diff = [
        ticks_sorted[i] - ticks_sorted[i - 1]
        for i in range(1, len(ticks_sorted))
    ]

    text_diff = [
        text_sorted[i] - text_sorted[i - 1]
        for i in range(1, len(text_sorted))
    ]

    # 4. Remove outliers
    # Bounding boxes may shift by a few pixels
    # OCR may misread values (e.g., 10 -> 18)
    # therefore outlier rejection is still necessary
    ticks_diff = reject_outliers(np.array(ticks_diff), m=1)
    text_diff = reject_outliers(np.array(text_diff), m=1)

    # 5. Compute normalization ratio
    if len(ticks_diff) == 0 or np.array(ticks_diff).mean() == 0:
        return text_sorted, 0  # avoid division by zero

    normalize_ratio = (
        np.array(text_diff).mean() /
        np.array(ticks_diff).mean()
    )

    # Use the smallest tick as anchor point
    min_val = text_sorted[0]
    min_pixel = ticks_sorted[0]  # must correspond to sorted text

    return text_sorted, normalize_ratio, (min_val, min_pixel)


### 3. Save Results

Export extracted values to:

- Excel file
- JSON file

Output format:
- image
- legend
- x_label
- value

In [41]:
def save_results(df: pd.DataFrame,
                 yValueDict: dict,
                 excel_path: str | Path,
                 json_path: str | Path):
    excel_path = Path(excel_path)
    json_path  = Path(json_path)

    excel_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.parent.mkdir(parents=True, exist_ok=True)

    # ---- 1) Save Excel ----
    # engine xlsxwriter
    df.to_excel(excel_path, index=False, engine="xlsxwriter")

    # ---- 2) Save JSON ----
    payload = {
        "meta": {
            "n_rows": int(len(df)),
            "columns": list(df.columns),
        },
        "yValueDict": yValueDict,
        "records": df.to_dict(orient="records"),
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("✅ Saved Excel:", str(excel_path))
    print("✅ Saved JSON :", str(json_path))

## 4. Geometry Helper Functions

Utility functions for:

- distance computation
- angle computation
- rectangle distance
- matching bounding boxes


In [42]:
def euclidean(v1, v2):
    return sum((p - q) ** 2 for p, q in zip(v1, v2)) ** .5

def angle_between(p1, p2):

    deltaX = p1[0] - p2[0]
    deltaY = p1[1] - p2[1]

    return math.atan2(deltaY, deltaX) / math.pi * 180

def RectDist(rectA, rectB):
    (rectAx, rectAy, rectAw, rectAh) = rectA
    (rectBx, rectBy, rectBw, rectBh) = rectB

    return abs(rectAx + rectAx / 2 - rectBx - rectBx / 2)

## 2. Section for Legend Analysis Task

### 2.1. Legend Patch Assignment

Match legend text with color patches detected by YOLO.

Method:
- compute cost matrix
- Hungarian algorithm
- match closest patch to each legend

In [43]:
import numpy as np
from scipy.optimize import linear_sum_assignment  # requires scipy

BIG_COST = 1e6


def assign_legend_patches(
    legend_boxes,
    patch_rects,
    y_tol=20,        # tolerance in Y (pixels) between legend center and patch center
    prefer_left=True,
    max_cost=None,   # if None: accept all valid pairs; otherwise reject pairs with cost > max_cost
):
    """
    Parameters
    ----------
    legend_boxes : list[(text, (tx, ty, tw, th))]
    patch_rects  : list[(x, y, w, h)] from YOLO detection

    Returns
    -------
    mapping : list of length = len(legend_boxes)
              each element is (x, y, w, h) or None if no valid patch is found
    """
    nL = len(legend_boxes)
    nP = len(patch_rects)

    if nL == 0 or nP == 0:
        return [None] * nL

    # Cost matrix (nLegend x nPatch)
    cost = np.full((nL, nP), BIG_COST, dtype=np.float32)

    for i, (_, (tx, ty, tw, th)) in enumerate(legend_boxes):
        cx_L = tx + tw / 2.0
        cy_L = ty + th / 2.0

        for j, (x, y, w, h) in enumerate(patch_rects):
            cx_P = x + w / 2.0
            cy_P = y + h / 2.0

            dy = abs(cy_L - cy_P)

            # Same-row constraint: if Y difference is too large, mark as invalid
            if dy > y_tol:
                continue  # keep cost = BIG_COST

            dx = cx_L - cx_P

            if prefer_left:
                # Default assumption: patch is to the left of legend (cx_P < cx_L)
                if dx <= 0:
                    # patch is to the right or aligned → invalid
                    continue

                # cost: horizontal distance + small penalty on vertical offset
                dist = dx + 0.3 * dy
            else:
                # No left/right constraint: use Euclidean distance
                dist = float(np.hypot(dx, dy))

            cost[i, j] = dist

    # Solve assignment problem (minimize total cost)
    row_ind, col_ind = linear_sum_assignment(cost)

    # mapping[i] = patch_rects[j] or None
    mapping = [None] * nL

    for i, j in zip(row_ind, col_ind):
        c = float(cost[i, j])

        # Reject dummy or excessively large costs
        if c >= BIG_COST:
            continue
        if max_cost is not None and c > max_cost:
            continue

        mapping[i] = patch_rects[j]

    return mapping

### 2.2. Bar-legend Mapping

In [44]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import timm
from timm.data import resolve_model_data_config
from timm.data.transforms_factory import create_transform
from PIL import Image
import numpy as np  # required for isinstance(image, np.ndarray)

# =========================================================
# Helper for preprocessing bbox
# =========================================================

def shrink_legend_bbox(
    bbox,
    img_size,
    ratio: float = 0.12,
    min_px: int = 2,
    max_px: int = 3,
):
    """
    Shrink legend bounding box in both directions (~0.1–0.15 ratio).
    Each side is reduced by 1–3 pixels (if size is large enough)
    to avoid including borders or text.

    bbox: (x1, y1, x2, y2)
    img_size: (W, H) of the original image
    """
    x1, y1, x2, y2 = bbox
    W, H = img_size
    w = x2 - x1
    h = y2 - y1

    if w <= 2 or h <= 2:
        # Too small, do not shrink
        return x1, y1, x2, y2

    # Compute shrink amount based on ratio
    dx = w * ratio
    dy = h * ratio

    # Clamp into [min_px, max_px] and ensure bbox remains valid
    # Ensure at least 2 pixels remain in width/height
    max_dx_allowed = max(0, (w - 2) / 2)
    max_dy_allowed = max(0, (h - 2) / 2)

    if max_dx_allowed <= 0 or max_dy_allowed <= 0:
        return x1, y1, x2, y2

    dx = min(max(dx, min_px), max_px, max_dx_allowed)
    dy = min(max(dy, min_px), max_px, max_dy_allowed)

    x1_new = x1 + dx
    x2_new = x2 - dx
    y1_new = y1 + dy
    y2_new = y2 - dy

    # Clamp inside image boundaries
    x1_new = max(0, min(x1_new, W - 1))
    x2_new = max(0, min(x2_new, W))
    y1_new = max(0, min(y1_new, H - 1))
    y2_new = max(0, min(y2_new, H))

    if x2_new <= x1_new or y2_new <= y1_new:
        # If shrink becomes invalid, return original bbox
        return x1, y1, x2, y2

    return int(x1_new), int(y1_new), int(x2_new), int(y2_new)


def shrink_bar_bbox_vertical(
    bbox,
    img_size,
    ratio_x: float = 0.12,
    min_px: int = 2,
    max_px: int = 4,
    shrink_y_px: int = 0,
):
    """
    Shrink bounding box for vertical bars:

    - Primarily shrink along horizontal axis (x)
    - Vertical axis (y) is shrunk minimally (default 0–1 px)
      to avoid touching axis/ticks while preserving bar height

    bbox: (x1, y1, x2, y2)
    img_size: (W, H)
    """
    x1, y1, x2, y2 = bbox
    W, H = img_size
    w = x2 - x1
    h = y2 - y1

    if w <= 2 or h <= 2:
        return x1, y1, x2, y2

    # Shrink along x similar to legend
    dx = w * ratio_x
    max_dx_allowed = max(0, (w - 2) / 2)

    if max_dx_allowed > 0:
        dx = min(max(dx, min_px), max_px, max_dx_allowed)
    else:
        dx = 0

    # Very small shrink along y (default ~1px each side)
    dy = min(shrink_y_px, max(0, (h - 2) / 2))

    x1_new = x1 + dx
    x2_new = x2 - dx
    y1_new = y1 + dy
    y2_new = y2 - dy

    # Clamp inside image boundaries
    x1_new = max(0, min(x1_new, W - 1))
    x2_new = max(0, min(x2_new, W))
    y1_new = max(0, min(y1_new, H - 1))
    y2_new = max(0, min(y2_new, H))

    if x2_new <= x1_new or y2_new <= y1_new:
        return x1, y1, x2, y2

    return int(x1_new), int(y1_new), int(x2_new), int(y2_new)


def central_crop_to_size(patch_pil: Image.Image, target_size):
    """
    Centrally crop PIL patch to target_size (w_t, h_t)
    without exceeding patch dimensions.

    target_size: (w_target, h_target)
    """
    w_t, h_t = target_size
    w, h = patch_pil.size

    if w_t <= 0 or h_t <= 0:
        return patch_pil

    # Ensure crop does not exceed image size
    w_t = min(w_t, w)
    h_t = min(h_t, h)

    left = (w - w_t) // 2
    top = (h - h_t) // 2
    right = left + w_t
    bottom = top + h_t

    return patch_pil.crop((left, top, right, bottom))

### 2.2.1. Deep Feature Extraction Backbone

Load pretrained models for visual similarity:

Supported backbones:
- ResNet50
- EfficientNet
- CLIP ViT
- Swin Transformer

Used for:
legend ↔ bar color matching

In [45]:



# =========================================================
# 1) Backbone + transform loader
# =========================================================

class FeatureMapMeanPool(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        x = self.model(x)[0]          # [B, C, H, W]
        return x.mean(dim=(2, 3))     # global mean pooling -> [B, C]


def get_backbone(model_type: str, device: str | None = None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = None

    # -----------------------------
    # RESNET50 (mid-level feature map)
    # -----------------------------
    if model_type == "resnet50":
        base_model = timm.create_model(
            "resnet50",
            pretrained=True,
            features_only=True,
            out_indices=(1,)   # stage 2 (0-based) -> more sensitive to low/mid-level features
        )
        model = FeatureMapMeanPool(base_model)
        data_config = resolve_model_data_config(base_model)

    # -----------------------------
    # EFFICIENTNET (option A: final embedding)
    # -----------------------------
    elif model_type == "efficientnet_b0":
        model = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0       # return embedding vector (global pooled)
        )
        data_config = resolve_model_data_config(model)

    elif model_type == "efficientnet_b1":
        model = timm.create_model(
            "efficientnet_b1",
            pretrained=True,
            num_classes=0
        )
        data_config = resolve_model_data_config(model)

    # -----------------------------
    # EFFICIENTNET (option B: mid-level feature map)
    # call using model_type = "efficientnet_b0_mid"
    # -----------------------------
    elif model_type == "efficientnet_b0_mid":
        base_model = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            features_only=True,
            out_indices=(2,)    # try (1,) or (2,) for shallower/deeper features
        )
        model = FeatureMapMeanPool(base_model)
        data_config = resolve_model_data_config(base_model)

    elif model_type == "efficientnet_b1_mid":
        base_model = timm.create_model(
            "efficientnet_b1",
            pretrained=True,
            features_only=True,
            out_indices=(2,)
        )
        model = FeatureMapMeanPool(base_model)
        data_config = resolve_model_data_config(base_model)

    # -----------------------------
    # OTHER MODELS
    # -----------------------------
    elif model_type == "clip_vitb32":
        model = timm.create_model(
            "vit_base_patch32_clip_224.openai",
            pretrained=True,
            num_classes=0
        )
        data_config = resolve_model_data_config(model)

    elif model_type == "swin_tiny":
        model = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )
        data_config = resolve_model_data_config(model)

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    # Set evaluation mode
    model.eval()
    model.to(device)

    # Create preprocessing transform
    transform = create_transform(**data_config, is_training=False)

    return {
        "model": model,
        "transform": transform,
        "device": device,
    }

### 2.2.2 Patch Embedding Extraction

Extract feature vector from:

- legend color patch
- bar region

Steps:
1. crop patch
2. resize & normalize
3. forward backbone
4. L2 normalize

In [47]:
def extract_patch_embedding(
    image,
    bbox,
    backbone,
    kind: str = "generic",
    legend_ref_size: tuple[int, int] | None = None,
    debug_options: dict | None = None,
):
    """
    Extract embedding for a patch defined by bbox, with specific preprocessing
    for legend/bar in vertical bar charts.

    Parameters
    ----------
    image : np.ndarray or PIL.Image
        Original RGB image (H, W, 3). If numpy, it will be converted to PIL.Image.
    bbox : tuple (x1, y1, x2, y2)
        Bounding box coordinates in the original image (pixel space).
    backbone : dict
        Output dictionary returned from get_backbone().
    kind : str
        "legend"  -> shrink legend bbox (0.1-0.15, 1-3 px each side)
        "bar"     -> shrink horizontally, minimal vertical shrink
        "generic" -> use bbox without preprocessing
    legend_ref_size : (w_leg, h_leg) or None
        If kind="bar" and legend_ref_size is provided, central crop the bar patch
        to approximately match legend size before resizing (reduce scale mismatch).

    Returns
    -------
    emb : torch.Tensor
        L2-normalized embedding vector, shape = (D,)
        (returned on CPU for easier downstream processing)
    """
    model = backbone["model"]
    transform = backbone["transform"]
    device = backbone["device"]

    # Convert to PIL if image is numpy
    if isinstance(image, np.ndarray):
        # Assume RGB image (H, W, 3)
        image_pil = Image.fromarray(image.astype("uint8"))
    elif isinstance(image, Image.Image):
        image_pil = image
    else:
        raise TypeError("image must be a numpy array or PIL.Image")

    W, H = image_pil.size  # PIL format: (width, height)
    x1, y1, x2, y2 = bbox

    # Apply bbox preprocessing depending on patch type
    if kind == "legend":
        x1, y1, x2, y2 = shrink_legend_bbox((x1, y1, x2, y2), (W, H))
    elif kind == "bar":
        x1, y1, x2, y2 = shrink_bar_bbox_vertical((x1, y1, x2, y2), (W, H))
    else:
        # generic: keep original bbox
        pass

    # Clamp coordinates inside image boundaries
    x1 = max(0, min(x1, W - 1))
    x2 = max(0, min(x2, W))
    y1 = max(0, min(y1, H - 1))
    y2 = max(0, min(y2, H))

    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            f"Invalid bbox after preprocessing: {(x1, y1, x2, y2)} "
            f"for image size {(W, H)}"
        )

    # Initial crop
    patch = image_pil.crop((x1, y1, x2, y2))

    # If bar and legend reference size provided -> central crop to legend size
    if kind == "bar" and legend_ref_size is not None:
        patch = central_crop_to_size(patch, legend_ref_size)

    _debug_show_patch(debug_options, patch, kind, (x1, y1, x2, y2))

    # Apply transform (resize, normalize, ...) -> tensor CHW
    patch_tensor = transform(patch)  # shape: (C, H, W)
    patch_tensor = patch_tensor.unsqueeze(0).to(device)  # shape: (1, C, H, W)

    # Forward pass to extract features
    with torch.no_grad():
        feat = model(patch_tensor)  # shape: (1, D) or (1, D, ...)

        # Flatten if model returns extra spatial dimensions
        feat = feat.view(feat.size(0), -1)  # (1, D)

        # L2 normalize for cosine similarity
        feat = F.normalize(feat, p=2, dim=1)

    emb = feat.squeeze(0).cpu()  # (D,)
    return emb

### 2.2.3 Legend-to-Bar Matching

Assign each bar to a legend using cosine similarity.

similarity = legend_embedding @ bar_embedding

Each bar → best matching legend

In [48]:
# =========================================================
# 3) Match legend -> bar using cosine similarity
# =========================================================

def match_legend_to_bars(
    legend_embs: torch.Tensor,
    bar_embs: torch.Tensor
):
    """
    Match each legend to the bar with highest similarity.

    Assumes legend_embs and bar_embs are already L2-normalized.

    Parameters
    ----------
    legend_embs : torch.Tensor
        Tensor of shape (L, D), where L is number of legend patches.
    bar_embs : torch.Tensor
        Tensor of shape (B, D), where B is number of bar patches.

    Returns
    -------
    matches : dict
        {
          "legend_to_bar": list length L,
                           each element is the matched bar index (int),
          "scores": list length L,
                    cosine similarity score for each match (float),
          "similarity_matrix": torch.Tensor shape (L, B)
                               (useful for debugging/visualization)
        }
    """
    if legend_embs.ndim != 2 or bar_embs.ndim != 2:
        raise ValueError("legend_embs and bar_embs must be 2D tensors")

    if legend_embs.size(1) != bar_embs.size(1):
        raise ValueError("Embedding dim mismatch between legend and bar")

    # Ensure float32
    legend_embs = legend_embs.float()
    bar_embs = bar_embs.float()

    # Compute similarity matrix: (L, D) @ (D, B) = (L, B)
    # Since embeddings are L2-normalized, dot product = cosine similarity
    sim_matrix = torch.matmul(legend_embs, bar_embs.t())  # (L, B)

    # Select best bar for each legend
    scores, indices = torch.max(sim_matrix, dim=1)  # shape (L,)

    matches = {
        "legend_to_bar": indices.tolist(),
        "scores": scores.tolist(),
        "similarity_matrix": sim_matrix,  # keep tensor for debugging if needed
    }

    return matches

### 2.3. Debug Visualization

Draw debugging image showing:

- axes
- legend boxes
- bar boxes
- matching lines
- x-label association

Used for visual verification

In [49]:
def draw_debug_image(
    base_image_rgb,
    xaxis, yaxis,
    legend_patches,
    legend_text_boxes,
    bar_rects,
    x_label_rects,
    legend_for_bar,
    x_label_for_bar,
    save_path,
    # --- FLAGS TO ENABLE/DISABLE CONNECTION LINES (default True) ---
    draw_line_legend_text=True,  # connect: legend color patch -> legend text
    draw_line_bar_legend=True,   # connect: bar -> legend color patch
    draw_line_bar_xlabel=True    # connect: bar -> x-label
):
    """
    Draw debugging visualization with optional connection lines.
    """
    img = cv2.cvtColor(base_image_rgb.copy(), cv2.COLOR_RGB2BGR)

    COLOR_AXIS      = (0, 0, 0)
    COLOR_XLABEL    = (160, 160, 160)
    thickness = 1

    def rect_to_xyxy(rect):
        x, y, w, h = rect
        return int(x), int(y), int(x + w), int(y + h)

    def rect_center(rect):
        x, y, w, h = rect
        return int(x + w / 2.0), int(y + h / 2.0)

    # ===== Color palette =====
    palette = [
        (255, 0, 0), (0, 0, 255), (0, 255, 0), (0, 255, 255),
        (255, 0, 255), (255, 255, 0), (128, 0, 255), (0, 128, 255),
        (128, 255, 0), (255, 128, 0),
    ]

    def color_for_legend(idx):
        if idx is None or idx < 0:
            return (100, 100, 100)
        return palette[idx % len(palette)]

    # 1) Draw axes
    if xaxis is not None:
        x1, y1, x2, y2 = map(int, xaxis)
        cv2.line(img, (x1, y1), (x2, y2), COLOR_AXIS, thickness)

    if yaxis is not None:
        x1, y1, x2, y2 = map(int, yaxis)
        cv2.line(img, (x1, y1), (x2, y2), COLOR_AXIS, thickness)

    # 2) Draw background for all x-labels (gray first)
    for rect in x_label_rects or []:
        x1, y1, x2, y2 = rect_to_xyxy(rect)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLOR_XLABEL, thickness)

    # ===== Legend =====
    legend_count = min(len(legend_patches or []), len(legend_text_boxes or []))

    # 3) Draw legend patches and legend text
    for legend_idx in range(legend_count):
        c = color_for_legend(legend_idx)
        patch_rect = legend_patches[legend_idx]
        text_rect  = legend_text_boxes[legend_idx]

        # draw patch bbox
        p_x1, p_y1, p_x2, p_y2 = rect_to_xyxy(patch_rect)
        cv2.rectangle(img, (p_x1, p_y1), (p_x2, p_y2), c, thickness)

        # draw text bbox
        t_x1, t_y1, t_x2, t_y2 = rect_to_xyxy(text_rect)
        cv2.rectangle(img, (t_x1, t_y1), (t_x2, t_y2), c, thickness)

        # [FLAG] draw line: patch -> text
        if draw_line_legend_text:
            c_patch = rect_center(patch_rect)
            c_text  = rect_center(text_rect)
            cv2.line(img, c_patch, c_text, c, thickness)

    # 4) Draw bars and x-labels (re-color)
    n_bars = len(bar_rects or [])

    for i in range(n_bars):
        bar_rect = bar_rects[i]

        leg_idx = None
        if legend_for_bar is not None and i < len(legend_for_bar):
            leg_idx = legend_for_bar[i]

        c = color_for_legend(leg_idx)
        c_bar = rect_center(bar_rect)

        # draw bar bbox
        x1, y1, x2, y2 = rect_to_xyxy(bar_rect)
        cv2.rectangle(img, (x1, y1), (x2, y2), c, thickness)

        # [FLAG] draw line: bar -> legend patch
        if leg_idx is not None and 0 <= leg_idx < legend_count:
            if draw_line_bar_legend:
                patch_rect = legend_patches[leg_idx]
                c_patch = rect_center(patch_rect)
                cv2.line(img, c_bar, c_patch, c, thickness)

        # process corresponding x-label
        if x_label_for_bar is not None and i < len(x_label_for_bar):
            lbl_rect = x_label_for_bar[i]

            if lbl_rect is not None:
                c_lbl = rect_center(lbl_rect)

                # [FLAG] draw line: bar -> x-label
                if draw_line_bar_xlabel:
                    cv2.line(img, c_bar, c_lbl, c, thickness)

                # always redraw x-label bbox using same color as bar
                # even if line is disabled, color still indicates association
                lx1, ly1, lx2, ly2 = rect_to_xyxy(lbl_rect)
                cv2.rectangle(img, (lx1, ly1), (lx2, ly2), c, thickness)

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, img)

    return img

## 3. Main Extraction Pipeline

Full processing pipeline for each image:

1. YOLO detect objects
2. detect axes
3. extract text labels
4. compute ratio
5. match legend patches
6. extract embeddings
7. match legend to bars
8. compute bar values
9. store results

In [59]:

images = []
texts = []

def _normalize_debug_options(debug_options: dict | None = None):
    cfg = {
        "enabled": False,
        "print_logs": True,
        "show_patch": False,
        "show_similarity": False,
        "save_overlay": False,
        "debug_dir": None,
    }
    if debug_options:
        cfg.update(debug_options)
    return cfg


def _debug_log(debug_options, message):
    if not debug_options:
        return
    if debug_options.get("enabled") and debug_options.get("print_logs", True):
        print(message)


def _debug_show_patch(debug_options, patch, kind, bbox_xyxy):
    if not debug_options:
        return
    if not (debug_options.get("enabled") and debug_options.get("show_patch")):
        return

    print(f"[debug patch][{kind}] bbox={bbox_xyxy}")
    plt.figure(figsize=(2, 2))
    plt.imshow(patch)
    plt.axis("off")
    plt.show()


def _debug_print_bar_similarity(
    debug_options,
    image_name,
    sim_matrix,
    legend_for_bar,
    legendtexts,
    bar_rects,
):
    if not debug_options:
        return
    if not (debug_options.get("enabled") and debug_options.get("show_similarity")):
        return
    if sim_matrix is None:
        return

    n_legends, n_bars = sim_matrix.shape
    print(f"[debug similarity] {image_name}: L={n_legends}, B={n_bars}")
    for b_idx, (bx, by, bw, bh) in enumerate(bar_rects):
        if b_idx >= len(legend_for_bar):
            continue

        best_legend_idx = legend_for_bar[b_idx]
        if best_legend_idx >= len(legendtexts):
            continue

        best_score = sim_matrix[best_legend_idx, b_idx].item()
        print(
            f"  Bar[{b_idx}] bbox=({bx}, {by}, {bw}, {bh}) -> "
            f"Legend[{best_legend_idx}] '{legendtexts[best_legend_idx]}', score={best_score:.4f}"
        )


def _debug_save_overlay(
    debug_options,
    image_rgb,
    xaxis,
    yaxis,
    legendrects,
    legend_text_rects,
    bar_rects,
    x_tick_list,
    legend_for_bar,
    text_boxes,
    image_stem,
):
    if not debug_options:
        return
    if not (debug_options.get("enabled") and debug_options.get("save_overlay")):
        return

    debug_dir = debug_options.get("debug_dir")
    if not debug_dir:
        return

    debug_path = os.path.join(debug_dir, f"{image_stem}_debug.png")
    x_label_rects = [box for (_, box) in x_tick_list]

    draw_debug_image(
        base_image_rgb=image_rgb,
        xaxis=xaxis,
        yaxis=yaxis,
        legend_patches=legendrects,
        legend_text_boxes=legend_text_rects,
        bar_rects=bar_rects,
        x_label_rects=x_label_rects,
        legend_for_bar=legend_for_bar,
        x_label_for_bar=text_boxes,
        save_path=debug_path,
    )
    _debug_log(debug_options, f"[debug] saved overlay: {debug_path}")


# 1. YOLO detect objects
def yolo_detect_objects(result):
    boxes = result.boxes
    names = result.names
    detections_by_class = {name: [] for name in names.values()}

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
        cls_id = int(box.cls[0].item())
        conf = float(box.conf[0].item())
        cls_name = names[cls_id]

        detections_by_class[cls_name].append(
            {
                "bbox": [x1, y1, x2, y2],
                "class_id": cls_id,
                "score": conf,
            }
        )

    legend_dets = detections_by_class.get("legend", [])
    bar_dets = detections_by_class.get("bar", [])
    plot_dets = detections_by_class.get("plot", [])
    return legend_dets, bar_dets, plot_dets


# 2. detect axes
def detect_axes(plot_dets):
    if not plot_dets:
        return None, None

    best_plot = max(
        plot_dets,
        key=lambda d: (d["bbox"][2] - d["bbox"][0]) * (d["bbox"][3] - d["bbox"][1]),
    )
    x1, y1, x2, y2 = map(int, best_plot["bbox"])
    xaxis = (x1, y2, x2, y2)
    yaxis = (x1, y1, x1, y2)
    return xaxis, yaxis


# 3. extract text labels
def extract_text_labels(image, task2_3, xaxis, yaxis):
    return getProbableLabels(image, task2_3, xaxis, yaxis)


# 4. compute ratio
def compute_ratio(y_tick_list):
    ratio_result = getRatio_optimized(y_tick_list)

    if len(ratio_result) == 3:
        return ratio_result

    list_text, normalize_ratio = ratio_result
    min_val = list_text[0] if list_text else 0.0
    min_pixel = 0.0
    return list_text, normalize_ratio, (min_val, min_pixel)


# 5. match legend patches
def match_legend_patches(legend_text_boxes, legend_dets, image_name, debug_options=None):
    legendtexts = []
    legendrects = []
    legend_text_rects = []

    if not legend_text_boxes:
        print(f"⚠️ Không tìm thấy legend text cho {image_name}, dùng 'series_0'.")
        return ["series_0"], legendrects, legend_text_rects

    legend_patch_boxes = []
    for det in legend_dets:
        x1, y1, x2, y2 = det["bbox"]
        legend_patch_boxes.append((x1, y1, x2 - x1, y2 - y1))

    if len(legend_patch_boxes) != len(legend_text_boxes):
        raise ValueError(
            f"Legend count mismatch for {image_name}: "
            f"{len(legend_patch_boxes)} patches vs {len(legend_text_boxes)} texts."
        )

    assignments = assign_legend_patches(
        legend_boxes=legend_text_boxes,
        patch_rects=legend_patch_boxes,
        y_tol=20,
        prefer_left=True,
        max_cost=None,
    )

    for idx, box in enumerate(legend_text_boxes):
        text, (textx, texty, width, height) = box
        patch_box = assignments[idx]
        if patch_box is None:
            _debug_log(debug_options, f"[warn] No patch near legend '{text}' in {image_name}.")
            continue

        legendrects.append(patch_box)
        legendtexts.append(text)
        legend_text_rects.append((textx, texty, width, height))
        _debug_log(debug_options, f"[legend] '{text}' -> bbox {patch_box}")

    if not legendtexts:
        print(f"⚠️ Không ghép được legend patch/text cho {image_name}, fallback 'series_0'.")
        return ["series_0"], [], []

    return legendtexts, legendrects, legend_text_rects


# 6. extract embeddings
def extract_embeddings(
    actual_image,
    legendrects,
    bar_rects,
    backbone,
    img_width,
    img_height,
    image_name,
    debug_options=None,
):
    legend_embs_list = []
    legend_sizes = []

    for (lx, ly, lw, lh) in legendrects:
        bbox_xyxy = (lx, ly, lx + lw, ly + lh)
        try:
            sx1, sy1, sx2, sy2 = shrink_legend_bbox(bbox_xyxy, (img_width, img_height))
            legend_sizes.append((sx2 - sx1, sy2 - sy1))

            emb = extract_patch_embedding(
                actual_image,
                bbox_xyxy,
                backbone,
                kind="legend",
                legend_ref_size=None,
                debug_options=debug_options,
            )
            legend_embs_list.append(emb)
        except Exception as e:
            print(f"⚠ Lỗi extract embedding legend patch {bbox_xyxy} trong {image_name}: {e}")

    legend_ref_size = None
    if legend_sizes:
        avg_w = int(sum(w for w, _ in legend_sizes) / len(legend_sizes))
        avg_h = int(sum(h for _, h in legend_sizes) / len(legend_sizes))
        legend_ref_size = (avg_w, avg_h)

    bar_embs_list = []
    for (bx, by, bw, bh) in bar_rects:
        bbox_xyxy = (bx, by, bx + bw, by + bh)
        try:
            emb = extract_patch_embedding(
                actual_image,
                bbox_xyxy,
                backbone,
                kind="bar",
                legend_ref_size=legend_ref_size,
                debug_options=debug_options,
            )
            bar_embs_list.append(emb)
        except Exception as e:
            print(f"⚠ Lỗi extract embedding bar patch {bbox_xyxy} trong {image_name}: {e}")

    return legend_embs_list, bar_embs_list


# 7. match legend to bars
def match_legend_to_bars_pipeline(
    legend_embs_list,
    bar_embs_list,
    legendtexts,
    bar_rects,
    image_name,
    debug_options=None,
):
    legend_for_bar = None
    sim_matrix = None

    if legend_embs_list and bar_embs_list and len(legend_embs_list) == len(legendtexts):
        try:
            legend_embs = torch.stack(legend_embs_list)
            bar_embs = torch.stack(bar_embs_list)
            matches = match_legend_to_bars(legend_embs, bar_embs)
            sim_matrix = matches["similarity_matrix"]
            legend_for_bar = torch.argmax(sim_matrix, dim=0).tolist()
            _debug_print_bar_similarity(
                debug_options,
                image_name,
                sim_matrix,
                legend_for_bar,
                legendtexts,
                bar_rects,
            )
        except Exception as e:
            print(f"⚠ Lỗi match_legend_to_bars trong {image_name}: {e}")
            legend_for_bar = None

    if legend_for_bar is None:
        if len(legendtexts) == 1:
            legend_for_bar = [0] * len(bar_rects)
        else:
            print(
                f"⚠ {image_name}: match_legend_to_bars không dùng được, "
                "tạm thời gán tất cả bar cho legend đầu tiên."
            )
            legend_for_bar = [0] * len(bar_rects)

    return legend_for_bar, sim_matrix


# 8. compute bar values
def compute_bar_values(
    legendtexts,
    x_tick_list,
    y_tick_list,
    bar_rects,
    legend_for_bar,
    normalize_ratio,
    min_val,
    debug_options=None,
):
    if not legendtexts:
        legendtexts = ["series_0"]

    text_boxes = []
    labels = []

    for rect_box in bar_rects:
        min_distance = sys.maxsize
        closest_box = None
        label_text = None

        for text, text_box in x_tick_list:
            d = RectDist(rect_box, text_box)
            if d < min_distance:
                min_distance = d
                closest_box = text_box
                label_text = text

        text_boxes.append(closest_box)
        labels.append(label_text)

    bar_heights = [(rect, float(rect[3])) for rect in bar_rects]
    nd = infer_ndigits_from_ticks(y_tick_list, default=1, cap=3)
    y_values = [(rect, round(min_val + h * normalize_ratio, nd + 1)) for rect, h in bar_heights]

    is_flat_mode = len(x_tick_list) == 0
    if is_flat_mode:
        data = {legend_text: 0.0 for legend_text in legendtexts}
    else:
        data = {
            legend_text: {x_label: 0.0 for x_label, _ in x_tick_list}
            for legend_text in legendtexts
        }

    for legend_idx, legendtext in enumerate(legendtexts):
        _debug_log(debug_options, f"[value] Assign values for legend '{legendtext}'")

        if is_flat_mode:
            found_val = 0.0
            for idx_bar, item in enumerate(y_values):
                if idx_bar < len(legend_for_bar) and legend_for_bar[idx_bar] == legend_idx:
                    found_val = item[1]
                    break
            data[legendtext] = found_val
            continue

        for x_label, box in x_tick_list:
            x, y, w, h = box
            value = 0.0
            dist = sys.maxsize

            for idx_bar, item in enumerate(y_values):
                if idx_bar >= len(legend_for_bar):
                    continue
                if legend_for_bar[idx_bar] != legend_idx:
                    continue
                if labels[idx_bar] != x_label:
                    continue

                vx, vy, vw, vh = item[0]
                cx_bar = vx + vw / 2.0
                cx_lbl = x + w / 2.0
                d = abs(cx_lbl - cx_bar)
                if d < dist:
                    dist = d
                    value = item[1]

            data[legendtext][x_label] = value

    return data, text_boxes


def getYVal(IMG_DIR, JSON_DIR, OUT_DIR, objects_dectector, backbone, debug_options=None):
    debug_options = _normalize_debug_options(debug_options)

    img_dir = Path(IMG_DIR)
    json_dir = Path(JSON_DIR)
    predict_dir = Path(OUT_DIR)

    image_paths = [p for p in img_dir.iterdir() if p.suffix.lower() in [".png", ".jpg", ".jpeg"]]
    results = run_inference(image_paths, predict_dir, objects_dectector)

    yValueDict = {}

    for index, path in enumerate(image_paths):
        if path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
            continue

        json_path = json_dir / f"{path.stem}.json"
        if not json_path.exists():
            print(f"⚠️ Không tìm thấy JSON cho ảnh {path.name}: {json_path}")
            continue

        try:
            task2_3 = json.loads(json_path.read_text(encoding="utf-8"))
            image = cv2.imread(str(path))
            if image is None:
                print(f"❌ Không đọc được ảnh {path.name}, bỏ qua.")
                continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            img_height, img_width, _ = image.shape
            actual_image = Image.open(path).convert("RGB")

            legend_dets, bar_dets, plot_dets = yolo_detect_objects(results[index])
            xaxis, yaxis = detect_axes(plot_dets)
            if xaxis is None or yaxis is None:
                print(f"❌ Không detect được plot cho {path.name}, bỏ qua.")
                continue

            (
                image,
                x_tick_list,
                x_title,
                y_tick_list,
                y_title,
                legend_text_boxes,
                image_text,
            ) = extract_text_labels(image, task2_3, xaxis, yaxis)

            list_text, normalize_ratio, (min_val, min_pixel) = compute_ratio(y_tick_list)
            if normalize_ratio is None:
                print(f"❌ Không tính được normalize_ratio cho {path.name}, bỏ qua chart này.")
                continue

            _debug_log(
                debug_options,
                f"[{index}] path: {path.name}, ratio: {normalize_ratio}",
            )

            legendtexts, legendrects, legend_text_rects = match_legend_patches(
                legend_text_boxes,
                legend_dets,
                path.name,
                debug_options=debug_options,
            )

            bar_rects = []
            for det in bar_dets:
                x1, y1, x2, y2 = det["bbox"]
                bar_rects.append((x1, y1, x2 - x1, y2 - y1))

            if not bar_rects:
                print(f"❌ Không phát hiện bar nào cho {path.name}, bỏ qua chart.")
                continue

            legend_embs_list, bar_embs_list = extract_embeddings(
                actual_image,
                legendrects,
                bar_rects,
                backbone,
                img_width,
                img_height,
                path.name,
                debug_options=debug_options,
            )

            legend_for_bar, sim_matrix = match_legend_to_bars_pipeline(
                legend_embs_list,
                bar_embs_list,
                legendtexts,
                bar_rects,
                path.name,
                debug_options=debug_options,
            )

            data, text_boxes = compute_bar_values(
                legendtexts,
                x_tick_list,
                y_tick_list,
                bar_rects,
                legend_for_bar,
                normalize_ratio,
                min_val,
                debug_options=debug_options,
            )

            _debug_save_overlay(
                debug_options,
                image,
                xaxis,
                yaxis,
                legendrects,
                legend_text_rects,
                bar_rects,
                x_tick_list,
                legend_for_bar,
                text_boxes,
                path.stem,
            )

            yValueDict[path.name] = data

        except Exception as e:
            print(f"❌ Lỗi khi xử lý {path.name}: {e}")
            continue

    return yValueDict


In [52]:
# ==== CONFIG ====
USE_COLAB = True

if USE_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/data_ex_v7/dataset")
    WEIGHT_PATH = "/content/drive/MyDrive/data_ex_v7/yolov8s_exp4/weights/best.pt"
    OUT_DIR = Path("/content/drive/MyDrive/data_ex_v7/dataset/outputs")
else:
    BASE_DIR = Path("D:/Projects/.../dataset")
    WEIGHT_PATH = "D:/Projects/.../weights/best.pt"
    OUT_DIR = Path("D:/Projects/.../outputs")
# ==================================
IMG_DIR  = BASE_DIR / "images_debug_1"
JSON_DIR = BASE_DIR / "json"
EXCEL_PATH = OUT_DIR / "y_values_v9.xlsx"
JSON_PATH  = OUT_DIR / "y_values_v9.json"
# ======================





### 5. YOLO Inference

Run object detection model to detect:

- bars
- legends
- plot region

In [58]:
def run_inference(img_dir, output_dir, detector):
    print(f"Running inference on images in: img_dir")
    results = detector.predict(
        source=img_dir,
        imgsz=832,
        conf=0.25,
        iou=0.5,
        save=True,
        project=output_dir,
        name="inference_YOLO",
        device=0
    )
    print("Inference done!")
    return results

In [60]:


# Load the model
objects_dectector = YOLO(WEIGHT_PATH)

# initial backbone

backbone = get_backbone("resnet50")  # or "clip_vitb32", "swin_tiny", "efficientnet_b0", "efficientnet_b0_mid", "resnet50"

debug_options = {
    "enabled": True,
    "print_logs": False,
    "show_patch": False,
    "show_similarity": False,
    "save_overlay": True,
    "debug_dir": str(OUT_DIR / "debug_viz"),
}
yValueDict = getYVal(IMG_DIR, JSON_DIR, OUT_DIR, objects_dectector, backbone, debug_options=debug_options)

# yValueDict = getYVal(IMG_DIR, JSON_DIR, objects_dectector, backbone)

rows = []
for img_name, legends_dict in yValueDict.items():
    for legend, legend_data in legends_dict.items():
        if isinstance(legend_data, dict):
            # Matrix mode: legend_data is a dictionary mapping x_label to value
            for x_label, val in legend_data.items():
                rows.append({
                    "image": img_name,
                    "legend": legend,
                    "x_label": x_label,
                    "value": float(val)  # ép float cho ổn định
                })
        else:
            # Flat mode: legend_data is the value itself
            rows.append({
                "image": img_name,
                "legend": legend,
                "x_label": "Value", # Placeholder for flat mode
                "value": float(legend_data)  # ép float cho ổn định
            })

df = pd.DataFrame(rows)

save_results(
    df=df,
    yValueDict=yValueDict,
    excel_path=EXCEL_PATH,
    json_path=JSON_PATH
)


Running inference on images in: img_dir

0: 832x832 2 legends, 8 bars, 1 plot, 23.0ms
1: 832x832 5 legends, 5 bars, 1 plot, 23.0ms
2: 832x832 3 legends, 9 bars, 1 plot, 23.0ms
Speed: 4.3ms preprocess, 23.0ms inference, 1.1ms postprocess per image at shape (1, 3, 832, 832)
Results saved to /content/drive/MyDrive/data_ex_v7/dataset/outputs/inference_YOLO
Inference done!
✅ Saved Excel: /content/drive/MyDrive/data_ex_v7/dataset/outputs/y_values_v9.xlsx
✅ Saved JSON : /content/drive/MyDrive/data_ex_v7/dataset/outputs/y_values_v9.json
